# Recommendation Systems and Market Basket Analysis in Online Retail: A Machine Learning Approach



## Introduction

Online retail businesses generate large volumes of transactional data that can be used to understand shopping behaviour and support better commercial decisions. Two of the most common machine learning applications in this domain are recommendation systems and market basket analysis. 
This report uses a real-world retail dataset covering a two-year period of customer transactions from the United Kingdom. The data was cleaned and prepared to remove invalid entries such as refunds, missing customer identifiers and non–product codes. After this preparation, two machine learning pipelines were implemented: collaborative filtering for recommendation systems, and Apriori and FP-Growth for association rule mining. The results provide insights into purchasing patterns and potential business value for a modern online retailer.

## General Objectives

- To implement and evaluate machine learning techniques that support decision-making in an online retail context.
- To explore how recommendation systems and association rule mining can reveal hidden patterns in customer purchasing behaviour.
- To examine the strengths, limitations and business implications of different machine learning models applied to retail data.

## Specific Objectives

1. To build a user–item interaction matrix from real transaction data and implement collaborative filtering models (user–user and item–item) for personalised recommendations.
2. To compare collaborative filtering with content-based filtering from a conceptual point of view.
3. To prepare transaction-level data for market basket analysis and apply Apriori and FP-Growth algorithms to identify frequent itemsets and association rules.
4. To compare the outputs of Apriori and FP-Growth, analysing similarities, differences and computational efficiency.
5. To produce interpretable insights that demonstrate how machine learning models can support product recommendations, cross-selling strategies and business optimisation.
6. To prepare visualisations justify the data preparation process used to build a clear and accessible dashboard.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity  
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules




df = pd.read_csv(
    "online_retail_II.csv",
    encoding="utf-8",
    low_memory=False)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
df.nunique()

### Basic cleaning 

In [ ]:
df.columns = df.columns.str.strip().str.replace(" ", "_")

In [ ]:
#    Convert InvoiceDate to datetime
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")

In [ ]:
df["Customer_ID"] = df["Customer_ID"].astype("Int64")

In [ ]:
 #Drop rows with invalid dates (if any)
df = df.dropna(subset=["InvoiceDate"])

In [ ]:
# Remove negative quantities (returns)
df = df[df["Quantity"] > 0]

In [ ]:
# Remove zero or negative prices (bad or invalid records)
df = df[df["Price"] > 0]

In [ ]:
target_country = "United Kingdom"   
df = df[df["Country"] == target_country]

In [ ]:
df = df.dropna(subset=["Customer_ID"])

In [ ]:
# Ensure Description is string (for safety)
df["Description"] = df["Description"].astype(str).str.strip()
df["StockCode"] = df["StockCode"].astype(str).str.strip()

In [ ]:
#  Remove StockCodes that appear only once in the filtered dataset
stock_counts = df["StockCode"].value_counts()
valid_stocks = stock_counts[stock_counts > 1].index
df = df[df["StockCode"].isin(valid_stocks)]

In [ ]:
# Remove Descriptions that appear only once in the filtered dataset
desc_counts = df["Description"].value_counts()
valid_desc = desc_counts[desc_counts > 1].index
df = df[df["Description"].isin(valid_desc)]

In [ ]:
df = df.reset_index(drop=True)

## Data Preparation

The dataset used in this project comes from a real online retail environment and covers customer transactions from December 2009 to December 2011. As with most operational datasets, the raw data contained noise, inconsistencies and entries that were not suitable for machine learning. Preparing the data correctly was essential to ensure that the recommendation system and the market basket analysis produced meaningful and reliable results.

### 1. Initial structure of the dataset
The raw dataset consisted of over 1.06 million rows and eight variables: Invoice, StockCode, Description, Quantity, InvoiceDate, Price, Customer ID and Country. Although structurally complete, several issues common in transactional data were detected:
- presence of negative quantities corresponding to returns or cancellations;
- missing values in the Customer ID field;
- StockCodes that referred to shipping charges, tests or adjustments rather than actual products;
- multiple countries included, which introduced geographical noise.

### 2. Filtering for the United Kingdom
Since the United Kingdom accounted for the vast majority of transactions, and the goal was to analyse consistent purchasing behaviour, only UK transactions were retained. This avoided heterogeneity caused by differing commercial conditions, seasonal calendars and cultural preferences across countries. The dataset remained large after filtering, maintaining statistical richness.

### 3. Handling missing and invalid values
Entries with missing Customer ID were removed because collaborative filtering requires identifiable users. Negative quantities were also removed, as they corresponded to returns rather than purchases. Both decisions ensured that the models reflected only positive purchasing behaviour.

### 4. Cleaning StockCode values
Some StockCodes did not correspond to physical products. Examples include:
- `POST` (postage),
- `BANK CHARGES`,
- `ADJUST2`,
- codes used for testing or internal adjustments.

These codes were retained in the raw dataset to preserve transactional integrity but excluded from the recommendation system stage, since recommending shipping fees or bank charges would be inappropriate. The main product catalogue remained intact.

### 5. Construction of specialised datasets
To support the different analytical tasks required in the assignment, the cleaned dataset was transformed into three purpose-built subsets:

- **df_rec**  
  Used for the recommendation system.  
  Contains: Customer_ID, StockCode and Interaction (total quantity purchased).  
  Aggregated at user–product level.

- **df_basket**  
  Used for market basket analysis (Apriori and FP-Growth).  
  Contains only Invoice and StockCode, allowing each invoice to be transformed into a list of items.

- **df_dash**  
  Used for the dashboard.  
  Includes enriched variables such as total revenue, year and month extracted from InvoiceDate to support descriptive visualisations.

This modular approach ensured that each machine learning task received a dataset optimised for its requirements without altering the original cleaned dataset.

### 6. Rationale for the preparation decisions
The data preparation strategy prioritised three principles:

1. **Consistency** – removing noise such as returns and non-product codes ensured cleaner behavioural patterns.
2. **Interpretability** – keeping only well-defined users and products made the outputs easier to interpret and justify.
3. **Model validity** – collaborative filtering requires user identifiers, and market basket analysis requires clean transaction lists. Each preparation step supported these methodological needs.

Overall, the data preparation process resulted in a stable, reliable and context-appropriate dataset on which the machine learning models could operate effectively.


# Dataset for Recommendation System (user–item)

In [ ]:
df_rec = df[["Customer_ID", "StockCode", "Quantity"]].copy()

# Quick sanity checks
print(df_rec.shape)

df_rec.head()

In [ ]:
# Aggregate quantity per (user, item)
df_rec = (df_rec.groupby(["Customer_ID", "StockCode"], as_index=False)["Quantity"]
    .sum())

# Rename quantity as an interaction strength if you want
df_rec = df_rec.rename(columns={"Quantity": "Interaction"})
df_rec.head()
print(df_rec.columns)

In [ ]:
n_users = df_rec["Customer_ID"].nunique()
n_items = df_rec["StockCode"].nunique()

print("Number of users:", n_users)
print("Number of items:", n_items)

In [ ]:
df_rec.to_csv("df_rec_user_item.csv", index=False)

In [ ]:
# Create user–item matrix from df_rec
user_item_matrix = df_rec.pivot_table(
    index="Customer_ID",   # = users
    columns="StockCode",   # = items
    values="Interaction",  # = interaction strength
    fill_value=0
)


user_item_matrix.head()

In [ ]:
user_item_matrix.columns

In [ ]:
#List of non-product / service codes we want to remove
service_codes = ['POST', 'BANK CHARGES', 'D', 'M', 'DOT',
                 'ADJUST2', 'PADS', 'C2', 'SP1002', 'TEST001']

In [ ]:
service_codes_in_matrix = [c for c in service_codes if c in user_item_matrix.columns]
print("Service codes found in matrix:", service_codes_in_matrix)

In [ ]:
# Create a clean version of the matrix without those columns
user_item_matrix_clean = user_item_matrix.drop(columns=service_codes_in_matrix)

In [ ]:
df[df["StockCode"].isin(service_codes)][["StockCode", "Description"]].drop_duplicates().head(50)


In [ ]:
[c for c in service_codes if c in user_item_matrix_clean.columns]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute cosine similarity between items
item_similarity = cosine_similarity(user_item_matrix_clean.T)

# Convert to a DataFrame for easier handling
item_sim_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix_clean.columns,
    columns=user_item_matrix_clean.columns)

item_sim_df.head()

In [ ]:
# Compute cosine similarity between users
user_similarity = cosine_similarity(user_item_matrix_clean)

user_sim_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix_clean.index,   # Customer_ID
    columns=user_item_matrix_clean.index  # Customer_ID
)

user_sim_df.head()

### Helper: item–item recommender for a user

In [ ]:
def recommend_items_user_based(customer_id, top_n=5, filter_seen=True):
    """
    User–user collaborative filtering recommender.

    If filter_seen=True (default), items already purchased by the user
    are removed from the recommendation list (discovery mode).

    If filter_seen=False, already purchased items can appear again.
    This is useful for evaluation with a held-out item.
    """
    # Check that the user exists in the similarity matrix
    if customer_id not in user_sim_df.index:
        return []

    # Similarities to all other users
    sims = user_sim_df.loc[customer_id].sort_values(ascending=False)

    # Remove the user themself
    sims = sims.drop(customer_id, errors="ignore")

    # Take top similar users (you can adjust this number)
    top_users = sims.head(20).index.tolist()

    # Items purchased by similar users
    similar_purchases = df_rec[df_rec["Customer_ID"].isin(top_users)]

    # Aggregate interaction strength over items
    candidate_counts = (
        similar_purchases
        .groupby("StockCode")["Interaction"]
        .sum()
        .sort_values(ascending=False)
    )

    # Items already purchased by the target user
    user_items = df_rec[df_rec["Customer_ID"] == customer_id]["StockCode"].unique()

    # Optionally remove items already purchased by the user
    if filter_seen:
        candidate_counts = candidate_counts[~candidate_counts.index.isin(user_items)]

    # Return top-N item codes
    return candidate_counts.head(top_n).index.tolist()

In [ ]:
def recommend_items_item_based_for_user(customer_id, top_n=5, filter_seen=True):
    """
    Item–item collaborative filtering recommender at user level.

    For a given user, it aggregates item–item similarity scores
    over all items the user has purchased.

    If filter_seen=True, items already purchased by the user
    are removed from the recommendation list.
    """
    # Items purchased by the user
    user_items = df_rec[df_rec["Customer_ID"] == customer_id]["StockCode"].unique()

    if len(user_items) == 0:
        return []

    # Start with zero scores for all items
    scores = pd.Series(0.0, index=item_sim_df.columns)

    # Accumulate similarity scores from each item the user has
    for item in user_items:
        if item in item_sim_df.index:
            scores = scores.add(item_sim_df.loc[item], fill_value=0.0)

    # Optionally remove items the user already owns
    if filter_seen:
        scores = scores.drop(user_items, errors="ignore")

    # Sort by score and take top-N
    scores = scores.sort_values(ascending=False)
    top_items = scores.head(top_n).index.tolist()

    return top_items

### Evaluation function (hit-rate@K)

In [ ]:
import random

def evaluate_recommender_hit_rate(df_rec, recommender_func, 
                                  min_items=5, top_n=5, max_users=500):
    """
    Simple hit-rate@K evaluation.

    For each eligible user, we hold out one purchased item as 'test'
    and check if the recommender returns it in the top-K list.
    """
    # Users with at least `min_items` distinct items
    user_item_counts = df_rec.groupby("Customer_ID")["StockCode"].nunique()
    eligible_users = user_item_counts[user_item_counts >= min_items].index.tolist()

    if len(eligible_users) == 0:
        print("No users with enough interactions for evaluation.")
        return None

    # Sample users for evaluation
    random.seed(42)
    random.shuffle(eligible_users)
    eligible_users = eligible_users[:max_users]

    hits = 0
    total = 0

    for user in eligible_users:
        items = df_rec[df_rec["Customer_ID"] == user]["StockCode"].unique().tolist()

        # Choose one item as the test item
        test_item = random.choice(items)

        # Get recommendations for this user
        recs = recommender_func(user, top_n=top_n)

        if test_item in recs:
            hits += 1
        total += 1

    hit_rate = hits / total if total > 0 else 0
    return hit_rate

### Evaluate both models (using filter_seen=False for evaluation)

In [ ]:
# User–user CF (evaluation mode: filter_seen=False)
hit_user_user = evaluate_recommender_hit_rate(
    df_rec,
    recommender_func=lambda user, top_n: recommend_items_user_based(user, top_n=top_n, filter_seen=False),
    min_items=5,
    top_n=5,
    max_users=500
)

# Item–item CF (evaluation mode: filter_seen=False)
hit_item_item = evaluate_recommender_hit_rate(
    df_rec,
    recommender_func=lambda user, top_n: recommend_items_item_based_for_user(user, top_n=top_n, filter_seen=False),
    min_items=5,
    top_n=5,
    max_users=500
)

print(f"User–User CF hit-rate@5:  {hit_user_user:.3f}")
print(f"Item–Item CF hit-rate@5: {hit_item_item:.3f}")

### Improved Collaborative Filtering: Binary Interactions and Filtering Sparse Users/Items

In [ ]:
# Aggregate quantity per (Customer_ID, StockCode)
df_rec = (
    df.groupby(["Customer_ID", "StockCode"], as_index=False)["Quantity"]
      .sum()
)

# Rename Quantity to Interaction
df_rec = df_rec.rename(columns={"Quantity": "Interaction"})

# Convert Interaction to binary: 1 = purchased at least once
df_rec["Interaction"] = 1 

# Number of distinct items per user
user_counts = df_rec.groupby("Customer_ID")["StockCode"].nunique()

# Number of distinct users per item
item_counts = df_rec.groupby("StockCode")["Customer_ID"].nunique()

# Thresholds (can be tuned)
min_items_per_user = 5      # user must have bought at least 5 different items
min_users_per_item = 20     # item must have been bought by at least 20 users

active_users = user_counts[user_counts >= min_items_per_user].index
popular_items = item_counts[item_counts >= min_users_per_item].index

# Keep only active users and popular items
df_rec = df_rec[
    df_rec["Customer_ID"].isin(active_users)
    & df_rec["StockCode"].isin(popular_items)
]

print("Users after filtering:", df_rec["Customer_ID"].nunique())
print("Items after filtering:", df_rec["StockCode"].nunique())

In [ ]:
# User–item matrix with the filtered df_rec
user_item_matrix = df_rec.pivot_table(
    index="Customer_ID",
    columns="StockCode",
    values="Interaction",
    fill_value=0
)

# Optional: keep only rows/columns with some activity
user_item_matrix_clean = user_item_matrix.loc[
    (user_item_matrix.sum(axis=1) > 0),
    (user_item_matrix.sum(axis=0) > 0)
]

print("Shape of user_item_matrix_clean:", user_item_matrix_clean.shape)

from sklearn.metrics.pairwise import cosine_similarity

# User–user similarity
user_similarity = cosine_similarity(user_item_matrix_clean)
user_sim_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix_clean.index,
    columns=user_item_matrix_clean.index
)

# Item–item similarity
item_similarity = cosine_similarity(user_item_matrix_clean.T)
item_sim_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix_clean.columns,
    columns=user_item_matrix_clean.columns
)

user_sim_df.shape, item_sim_df.shape

### Re-run CF evaluation to compare

In [ ]:
# User–user CF (evaluation mode: filter_seen=False)
hit_user_user = evaluate_recommender_hit_rate(
    df_rec,
    recommender_func=lambda user, top_n: recommend_items_user_based(
        user,
        top_n=top_n,
        filter_seen=False
    ),
    min_items=5,
    top_n=5,
    max_users=500
)

# Item–item CF (evaluation mode: filter_seen=False)
hit_item_item = evaluate_recommender_hit_rate(
    df_rec,
    recommender_func=lambda user, top_n: recommend_items_item_based_for_user(
        user,
        top_n=top_n,
        filter_seen=False
    ),
    min_items=5,
    top_n=5,
    max_users=500
)

print(f"User–User CF hit-rate@5 (filtered):  {hit_user_user:.3f}")
print(f"Item–Item CF hit-rate@5 (filtered): {hit_item_item:.3f}")

The original hit-rate@5 values for both collaborative filtering models were around 0.08. After applying binary interactions and filtering very weak users and extremely rare items, the values increased to around 0.16 and 0.17. This shows that the models behave better when the interaction data is simplified and the sparsity of the matrix is reduced. Users with very few purchases and items with almost no overlap across users were adding noise and making the similarity calculations less meaningful. When those cases are removed, the similarity patterns become a bit clearer and the recommender can recover more of the held-out items. The improvement is not huge, but for this type of retail dataset it is already a noticeable change

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

navy       = "#0B1F3B"   
blue_main  = "#1F77B4"   
blue_light = "#A7C6ED"  
grey_grid  = "#D3D3D3" 
blue_soft  = "#A7C6ED" 

In [ ]:
# Compute item popularity by number of invoices
item_freq = (
    df.groupby("StockCode")["Invoice"]
      .nunique()
      .sort_values(ascending=False))

top_items = item_freq.head(12)
item_desc_map = (
    df[['StockCode', 'Description']]
    .dropna()
    .drop_duplicates()
    .set_index('StockCode')['Description']
    .to_dict())


top_labels = [
    item_desc_map.get(code, str(code)) 
    for code in top_items.index]

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(
    y=top_labels,
    width=top_items.values,
    color=blue_main)

ax.invert_yaxis()
for i, v in enumerate(top_items.values):
    ax.text(v + 20, i, str(v), va="center", fontsize=10, color=navy)
ax.set_title("Top 12 Most Purchased Products", fontsize=16, color=navy)
ax.set_xlabel("Number of Invoices", fontsize=13, color=navy)
ax.set_ylabel("Product Description", fontsize=13, color=navy)
ax.grid(axis="x", linestyle="--", color=grey_grid, alpha=0.5)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
top_codes = top_items.index.tolist()
co_matrix = pd.DataFrame(0, index=top_codes, columns=top_codes, dtype=int)
for inv, items in df.groupby("Invoice")["StockCode"]:
    items_list = [i for i in items if i in top_codes]
    unique_list = list(set(items_list))
    for i in range(len(unique_list)):
        for j in range(i + 1, len(unique_list)):
            a, b = unique_list[i], unique_list[j]
            co_matrix.loc[a, b] += 1
            co_matrix.loc[b, a] += 1

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(co_matrix.values, cmap="Blues")
ax.set_title("Co-Purchase Matrix – Top 12 Items", fontsize=14, color=navy)
ax.set_xticks(range(len(top_codes)))
ax.set_yticks(range(len(top_codes)))
ax.set_xticklabels(top_codes, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(top_codes, fontsize=9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Number of Co-Purchases", fontsize=10, color=navy)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Colors
navy       = "#102A43"
blue_main  = "#1F77B4"
blue_soft  = "#A7C6ED"
grey_grid  = "#D3D3D3"

# Make sure InvoiceDate is datetime
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# Create Year-Month period
df["YearMonth"] = df["InvoiceDate"].dt.to_period("M")

# Group by Year-Month and count unique invoices
monthly_counts = (
    df.groupby("YearMonth")["Invoice"]
      .nunique()
      .reset_index()
)

monthly_counts.columns = ["YearMonth", "Num_Invoices"]

# Keep only up to 2011-12
monthly_counts = monthly_counts[monthly_counts["YearMonth"] <= "2011-12"]

# Convert YearMonth (Period) to Timestamp for plotting
monthly_counts["Month"] = monthly_counts["YearMonth"].dt.to_timestamp()

# Plot
fig, ax = plt.subplots(figsize=(10, 4))

ax.fill_between(
    monthly_counts["Month"],
    monthly_counts["Num_Invoices"],
    color=blue_soft,
    alpha=0.3
)

ax.plot(
    monthly_counts["Month"],
    monthly_counts["Num_Invoices"],
    color=navy,
    linewidth=2,
    marker="o",
    markersize=4
)

ax.set_title("Monthly Transactions", fontsize=14, color=navy)
ax.set_xlabel("Month", fontsize=11, color=navy)
ax.set_ylabel("Unique Invoices", fontsize=11, color=navy)

ax.grid(True, axis="y", linestyle="--", color=grey_grid, alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.autofmt_xdate(rotation=30)
plt.tight_layout()
plt.show()

## Dataset for Recommendation System

Before building the recommendation models, it was important to understand the main interaction patterns in the dataset. The visuals included in this section provide a clear view of the products and transaction behaviour that drive the learning process of the system.

The chart of the top purchased products shows which items appear most frequently across invoices. These popular products offer strong and reliable interaction signals, and they help the models identify meaningful similarity between users and items.

The monthly transactions chart illustrates how activity evolved over time. The pattern is stable, with regular monthly activity, which indicates that the dataset contains consistent interaction behaviour rather than isolated spikes. This stability supports the use of collaborative filtering.

A small co-purchase matrix was also included for the most common products. It highlights which items tend to appear together in the same invoice. These co-occurrence relationships are one of the intuitive foundations of both collaborative filtering and market basket analysis.

Overall, this brief exploratory view of the dataset helps justify the modelling choices and provides context for how the recommendation system interprets purchasing patterns.

# Market Basket

In [ ]:
df_mba = df.copy()
df_mba = df_mba[df_mba["Quantity"] > 0]

# Remove cancelled invoices (prefix 'C')
if "Invoice" in df_mba.columns:
    df_mba = df_mba[~df_mba["Invoice"].astype(str).str.startswith("C")]
elif "InvoiceNo" in df_mba.columns:
    df_mba = df_mba[~df_mba["InvoiceNo"].astype(str).str.startswith("C")]

# drop missing customers if needed for later analysis
for col_name in ["Customer_ID", "CustomerID", "Customer ID"]:
    if col_name in df_mba.columns:
        df_mba = df_mba.dropna(subset=[col_name])
        break

df_mba.shape

In [ ]:
#  Build transactions
invoice_col = "Invoice" if "Invoice" in df_mba.columns else "InvoiceNo"
item_col    = "StockCode"
transactions = (
    df_mba
    .groupby(invoice_col)[item_col]
    .apply(lambda x: list(set(x.astype(str))))
    .tolist())
print(f"Number of transactions: {len(transactions)}")
print("Example transaction:", transactions[0][:10])

In [ ]:
# One-hot encode transactions
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
basket_df = pd.DataFrame(te_array, columns=te.columns_)
basket_df.head()

In [ ]:
# 4) Run FP-Growth on the reduced basket

from mlxtend.frequent_patterns import fpgrowth

min_support = 0.005  # 0.5%

frequent_itemsets_fp = fpgrowth(
    basket_df,
    min_support=min_support,
    use_colnames=True
)

frequent_itemsets_fp["itemset_size"] = frequent_itemsets_fp["itemsets"].apply(len)
frequent_itemsets_fp["itemset_str"] = frequent_itemsets_fp["itemsets"].apply(
    lambda x: ", ".join(sorted(list(x)))
)

frequent_itemsets_fp.head()


In [ ]:
# Association rules from FP-Growth itemsets
rules_fp = association_rules(frequent_itemsets_fp,metric="lift",min_threshold=1.0)
rules_fp["rule_length"] = rules_fp["antecedents"].apply(len) + rules_fp["consequents"].apply(len)
print(f"Number of rules extracted (FP-Growth): {len(rules_fp)}")
rules_fp.head()

In [ ]:
#  Filter useful rules
rules_filtered = rules_fp[
    (rules_fp["support"] >= 0.005) &
    (rules_fp["confidence"] >= 0.30) &
    (rules_fp["lift"] >= 1.50)
].sort_values("lift", ascending=False)

rules_filtered[["antecedents", "consequents", "support", "confidence", "lift"]].head(10)

In [ ]:
#  Map product codes to descriptions for human readable rules
item_desc_map = (
    df[["StockCode", "Description"]]
    .dropna()
    .drop_duplicates()
    .set_index("StockCode")["Description"]
    .to_dict())

def codes_to_descriptions(itemset):
    return [
        f"{code} — {item_desc_map.get(code, '(no description)')}"
        for code in itemset
    ]

def describe_rule(row):
    return pd.Series({
        "Antecedents": "; ".join(codes_to_descriptions(row["antecedents"])),
        "Consequents": "; ".join(codes_to_descriptions(row["consequents"])),
        "Support": row["support"],
        "Confidence": row["confidence"],
        "Lift": row["lift"]})
rules_final = rules_filtered.apply(describe_rule, axis=1)
rules_final.head(10)

The basket analysis helped me confirm how the dataset behaves at a transactional level, which was important for the assignment because it shows whether there are stable patterns that can support machine learning models. The rules came out clean and easy to interpret, and both Apriori and FP Growth pointed in the same direction even though they work differently. Their support, confidence and lift values were almost identical, which suggests that the structure of the data is strong enough to produce reliable associations no matter which algorithm is used. Most rules fall into the low-support but high-lift area, which is normal in retail datasets where many products sell in small volumes but still appear together in consistent ways. This matches the visual results, where the strongest rules come from families of products that customers naturally combine. Overall, the analysis confirmed that the transaction patterns are far from random and that the dataset contains enough repeated behaviour to justify the recommendation models and the rest of the machine learning process in the assignment.

### # Basic comparison between Apriori and FP-Growth rule sets

In [ ]:
comp_rows = []

if "rules" in globals():
    comp_rows.append({
        "Algorithm": "Apriori",
        "Num_rules": len(rules),
        "Mean_support": rules["support"].mean(),
        "Mean_confidence": rules["confidence"].mean(),
        "Mean_lift": rules["lift"].mean()
    })

if "rules_fp" in globals():
    comp_rows.append({
        "Algorithm": "FP-Growth",
        "Num_rules": len(rules_fp),
        "Mean_support": rules_fp["support"].mean(),
        "Mean_confidence": rules_fp["confidence"].mean(),
        "Mean_lift": rules_fp["lift"].mean()
    })

comparison_df = pd.DataFrame(comp_rows)
comparison_df

In [ ]:
rules_fp_final = rules_fp.apply(describe_rule, axis=1)
rules_fp_final.head(10)

In [ ]:
import plotly.express as px

lift_plot_df = []

if "rules" in globals():
    temp = rules[["lift"]].copy()
    temp["Algorithm"] = "Apriori"
    lift_plot_df.append(temp)

if "rules_fp" in globals():
    temp = rules_fp[["lift"]].copy()
    temp["Algorithm"] = "FP-Growth"
    lift_plot_df.append(temp)

lift_plot_df = pd.concat(lift_plot_df, ignore_index=True)

fig_lift = px.histogram(
    lift_plot_df,
    x="lift",
    color="Algorithm",
    barmode="overlay",
    nbins=50,
    opacity=0.6,
)

fig_lift.update_layout(
    title="Distribution of lift for Apriori vs FP-Growth",
    xaxis_title="Lift",
    yaxis_title="Number of rules",
)

fig_lift.show()

The Market Basket Analysis worked well for this dataset because both Apriori and FP Growth were able to uncover consistent buying patterns. Apriori produced slightly more rules than FP Growth, but the quality of the patterns was almost the same in terms of support, confidence and lift, which tells me that both methods were basically finding the same structure in how customers combine products. FP Growth did it in a more direct way, since it does not expand so many combinations, but the difference in the final rules was minimal.When looking at the numbers, the average support and confidence from both algorithms were nearly identical, and the lift values showed a similar distribution, with many rules around moderate lift and a few rules with very strong co purchase signals. This matches what we see in the sample rules, where items from the same style or collection appear together quite naturally. Customers tend to buy visually related items, or small complements, which confirms that the associations are not random noise but part of the actual retail behaviour in this dataset.

### Distribution of lift

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(rules_filtered["support"], bins=30, color=blue_light, edgecolor=navy, alpha=0.8)
plt.xlabel("Support", fontsize=12, color=navy)
plt.ylabel("Count", fontsize=12, color=navy)
plt.title("Distribution of Support Values", fontsize=14, color=navy)
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
rules_filtered["rule_len"] = (
    rules_filtered["antecedents"].apply(len) +
    rules_filtered["consequents"].apply(len)
)

plt.figure(figsize=(7,5))
rules_filtered["rule_len"].value_counts().sort_index().plot(
    kind="bar",
    color="blue",
    edgecolor=navy
)
plt.xlabel("Rule length", fontsize=12, color=navy)
plt.ylabel("Number of rules", fontsize=12, color=navy)
plt.title("Number of Rules by Size", fontsize=14, color=navy)
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
top_rules = rules_filtered.head(20)

clusters = {}

for _, row in top_rules.iterrows():
    ant = tuple(sorted(list(row["antecedents"])))
    cons = tuple(sorted(list(row["consequents"])))
    pair = ant + cons
    clusters.setdefault(pair, 0)
    clusters[pair] += 1

top_clusters = sorted(clusters.items(), key=lambda x: x[1], reverse=True)[:3]

top_clusters

In [ ]:
from collections import Counter

rules_use = rules_filtered.copy()
if rules_use.empty:
    rules_use = rules.copy()

all_items = []

for items in rules_use["antecedents"]:
    all_items.extend(list(items))
for items in rules_use["consequents"]:
    all_items.extend(list(items))

item_counts = Counter(all_items)
top_items = pd.Series(item_counts).sort_values(ascending=False).head(10)

labels = [item_desc_map.get(code, str(code)) for code in top_items.index]

plt.figure(figsize=(12, 6))
plt.barh(labels, top_items.values, color=blue_light, edgecolor=navy)
plt.gca().invert_yaxis()

plt.xlabel("Number of rules", fontsize=12, color=navy)
plt.title("Top 10 Products by Participation in Association Rules", fontsize=14, color=navy)
plt.grid(axis="x", linestyle="--", color=grey_grid, alpha=0.4)

for i, v in enumerate(top_items.values):
    plt.text(v + 0.5, i, str(v), va="center", fontsize=9, color=navy)

plt.tight_layout()
plt.show()

##  Conclusion

The analytical workflow combined data cleaning, exploratory inspection, recommendation modelling, and market basket analysis to reveal how customers behave and how products relate to each other within the retail dataset.
Despite the natural sparsity of transactional data, the collaborative filtering models (user–user and item–item) produced coherent similarity structures, and the evaluation showed that item-based predictions outperform user-based interactions. This indicates that product affinity is more stable than customer overlap in this dataset.

The market basket analysis complemented the recommendation work by identifying stable association rules with meaningful lift, confidence, and support. These rules revealed bundles, co-purchase behaviours, and product groupings that are not visible through collaborative filtering alone. The 3D rule landscape helped illustrate differences in rule frequency, reliability, and associative strength, making patterns easier to interpret.

Together, the models show that combining behavioural similarity (CF) with structural co-occurrence (MBA) provides a more complete understanding of how items interact in real purchasing contexts. This hybrid perspective strengthens recommendation strategies, supports product placement decisions, and enhances data-driven retail insights.

# Interactive Dashboard 

The dashboard summarises key aspects of the online retail dataset and visually supports 
why this data is suitable for Machine Learning models in an online retail business 


In [ ]:
 !pip install jupyter-dash dash plotly


In [ ]:
import plotly.express as px
from jupyter_dash import JupyterDash
from dash import dcc, html, Input, Output

In [ ]:
df_final = df.copy()

# Ensure InvoiceDate is datetime
if "InvoiceDate" in df_final.columns:
    df_final["InvoiceDate"] = pd.to_datetime(df_final["InvoiceDate"])
else:
    raise KeyError("No 'InvoiceDate' column found in df_final. Please check your column names.")

# Try to detect Quantity and Unit Price columns automatically
quantity_col = None
unit_price_col = None

for col in df_final.columns:
    col_lower = col.lower()
    if quantity_col is None and "quantity" in col_lower:
        quantity_col = col
    if unit_price_col is None and ("unitprice" in col_lower or col_lower == "price"):
        unit_price_col = col

print("Detected quantity column:", quantity_col)
print("Detected unit price column:", unit_price_col)

if quantity_col is None or unit_price_col is None:
    raise KeyError(
        f"Could not automatically detect quantity / price columns. "
        f"Columns available: {list(df_final.columns)}"
    )

# Create TotalSales
if "TotalSales" not in df_final.columns:
    df_final["TotalSales"] = df_final[quantity_col] * df_final[unit_price_col]

In [ ]:
import plotly.express as px

ACCENT_COLOR = "#ff6b6b"  # coral
BASE_COLORS = ["#4c6a9a", "#88a3d1", "#cfd8e6"]

def apply_senior_friendly_layout(fig, title_text):
    fig.update_layout(
        title=title_text,
        title_font_size=26,
        font=dict(size=18),
        legend=dict(font=dict(size=16)),
        margin=dict(l=60, r=40, t=80, b=60),
        hoverlabel=dict(font_size=16),
    )
    fig.update_xaxes(title_font=dict(size=20), tickfont=dict(size=16))
    fig.update_yaxes(title_font=dict(size=20), tickfont=dict(size=16))
    return fig

In [ ]:
import plotly.express as px

df_monthly = (
    df_final
    .set_index("InvoiceDate")
    .resample("M")["TotalSales"]
    .sum()
    .reset_index())

NAVY = "#001435"
fig_monthly = px.line(
    df_monthly,
    x="InvoiceDate",
    y="TotalSales",
    markers=True)
fig_monthly.update_traces(
    line=dict(width=4, color=NAVY),
    marker=dict(size=8, color=NAVY),
    hovertemplate="%{x|%b %Y}<br>Total sales: £%{y:,.0f}<extra></extra>"
)

fig_monthly = apply_senior_friendly_layout(
    fig_monthly,
    "Monthly Sales Trend — Stable Purchase Behaviour"
)

start_date = df_monthly["InvoiceDate"].min()
end_date = df_monthly["InvoiceDate"].max()

fig_monthly.update_layout(
    title=dict(
        text="Monthly Sales Trend — Stable Purchase Behaviour",
        font=dict(size=36, family="Trebuchet MS", color="#0f172a"),
        x=0.03
    ),
    width=900,
    height=820,
    hovermode="x unified",
    margin=dict(l=80, r=60, t=110, b=430),
    legend=dict(
        title_text="",
        font=dict(size=17),
        orientation="h",
        yanchor="bottom",
        y=1.03,
        xanchor="right",
        x=1
    ),
    annotations=[
        dict(
            x=start_date,
            yref="paper",
            y=-0.02,
            xref="x",
            showarrow=False,
            xanchor="left",
            font=dict(size=12, color="#4b5563"),
            text="Start of data"
        ),
        dict(
            x=end_date,
            yref="paper",
            y=-0.02,
            xref="x",
            showarrow=False,
            xanchor="right",
            font=dict(size=12, color="#4b5563"),
            text="End of data"
        ),
        dict(
            xref="paper",
            yref="paper",
            x=0.0,
            y=-0.70,
            yanchor="top",
            showarrow=False,
            align="left",
            font=dict(size=15, color="#4b5563"),
            text=(
                "Interaction tips:<br>"
                "• Hover to see the value for each month.<br>"
                "• Drag on the chart area to zoom in on a specific period.<br>"
                "• Use the time buttons or the slider below the axis to move through time.<br>"
                "• Double-click anywhere on the chart to reset the full view."
            )
        )
    ]
)

fig_monthly.update_xaxes(
    title_text="Month",
    title_standoff=40,
    rangeslider_visible=True,
    rangeselector=dict(
        buttons=[
            dict(count=1, label=" 1 m ", step="month", stepmode="backward"),
            dict(count=3, label=" 3 m ", step="month", stepmode="backward"),
            dict(count=1, label=" 1 y ", step="year", stepmode="backward"),
            dict(step="all", label=" All ")
        ],
        y=1.12,
        x=0.02,
        xanchor="left",
        yanchor="top",
        font=dict(size=15),
        bgcolor="#f1f5f9",
        activecolor="#e2e8f0"
    )
)

fig_monthly.update_yaxes(
    title_text="Total Sales (£)",
    tickprefix="£",
    separatethousands=True
)

fig_monthly.show()

In [ ]:
df_daily = (
    df_final
    .set_index("InvoiceDate")
    .resample("D")["TotalSales"]
    .sum()
    .reset_index()
)

fig_w6 = px.bar(
    df_daily,
    x="InvoiceDate",
    y="TotalSales",
    color_discrete_sequence=[ACCENT_COLOR],
)

fig_w6 = apply_senior_friendly_layout(
    fig_w6,
    "Daily sales with interactive time window"
)

fig_w6.update_layout(
    xaxis_title="Date",
    yaxis_title="Total sales",
)

fig_w6.update_xaxes(
    rangeslider_visible=True,
    rangeselector=dict(
        buttons=list([
            dict(count=1, label="1m", step="month", stepmode="backward"),
            dict(count=3, label="3m", step="month", stepmode="backward"),
            dict(count=6, label="6m", step="month", stepmode="backward"),
            dict(step="all", label="All"),
        ])
    )
)

fig_w6.show()


The increase in sales during the final quarter reflects Christmas-driven demand, while the apparent drop in January is explained by limited data availability at the end of the dataset.
This temporal context is essential to interpret all subsequent analyses.

In [ ]:
df_heat = df_final.copy()
df_heat["Day"] = df_heat["InvoiceDate"].dt.day_name()
df_heat["Hour"] = df_heat["InvoiceDate"].dt.hour

heat = (
    df_heat
    .groupby(["Day", "Hour"], as_index=False)["TotalSales"]
    .sum()
)

fig_heat = px.density_heatmap(
    heat,
    x="Hour",
    y="Day",
    z="TotalSales",
    color_continuous_scale=["#e2e8f0", "#12325c", "#0a1a33"]
)

fig_heat.update_layout(
    title="Sales Heatmap – Hour vs Day of Week",
    xaxis_title="Hour",
    yaxis_title="Day",
    title_font=dict(size=28, color="#0a1a33")
)

fig_heat.show()

The heatmap combines day of the week and hour of the day to show when purchasing activity concentrates.
Rather than focusing on totals, this visualisation highlights time windows where sales are more frequent.
Clear patterns emerge during specific hours on mid week days, which helps identify optimal moments for promotions and automated recommendation triggers.
Lower activity on certain days reflects the distribution of transactions in the dataset rather than the absence of demand.
This representation supports operational decisions related to timing, system activation, and campaign scheduling.

In [ ]:
def apply_senior_friendly_layout(fig, title_text):
    fig.update_layout(
        title=dict(
            text=title_text,
            font=dict(size=30, family="Trebuchet MS", color=NAVY_DARK),
            x=0.02
        ),
        paper_bgcolor="white",
        plot_bgcolor="white",
        font=dict(size=16, family="Trebuchet MS", color=NAVY_DARK),
        hoverlabel=dict(font_size=14),
        margin=dict(l=80, r=120, t=90, b=140)
    )
    return fig

dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

df_dow = df_final.copy()
df_dow["DayOfWeek"] = pd.Categorical(
    df_dow["InvoiceDate"].dt.day_name(),
    categories=dow_order,
    ordered=True
)

df_dow_agg = (
    df_dow
    .groupby("DayOfWeek", observed=False, as_index=False)["TotalSales"]
    .sum()
)

df_dow_agg = (
    df_dow_agg
    .set_index("DayOfWeek")
    .reindex(dow_order, fill_value=0)
    .reset_index()
)

fig_dow = px.bar(
    df_dow_agg,
    x="DayOfWeek",
    y="TotalSales",
    color_discrete_sequence=[ACCENT_COLOR]
)

fig_dow.update_traces(
    hovertemplate="Day: %{x}<br>Total sales: £%{y:,.0f}<extra></extra>",
    name="Total Sales"
)

fig_dow = apply_senior_friendly_layout(
    fig_dow,
    "Sales by Day of Week – Weekly Shopping Rhythm"
)

fig_dow.update_xaxes(
    title_text="Day of week",
    title_standoff=35,
    tickfont=dict(size=14, color=NAVY_DARK),
    categoryorder="array",
    categoryarray=dow_order
)

fig_dow.update_yaxes(
    title_text="Total Sales (£)",
    tickprefix="£",
    separatethousands=True,
    title_font=dict(color=NAVY_DARK),
    tickfont=dict(color=NAVY_DARK)
)

fig_dow.show()


This chart highlights the weekly rhythm of purchasing activity.
Mid-week days show stronger sales performance, which is relevant for scheduling promotions and system-driven recommendations.
Lower activity on Saturdays reflects the characteristics of the dataset rather than an absence of demand.
These patterns help identify optimal moments to activate marketing and recommendation strategies.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

NAVY_DARK = "#0a1a33"
NAVY_MED = "#12325c"
ACCENT_COLOR = NAVY_MED

df_products_sorted = (
    df_final
    .groupby("Description", as_index=False)["TotalSales"]
    .sum()
    .sort_values("TotalSales", ascending=False)
)

N_default = 10
df_top_products = df_products_sorted.head(N_default)

fig_top = px.bar(
    df_top_products,
    x="Description",
    y="TotalSales",
    color_discrete_sequence=[ACCENT_COLOR]
)

fig_top.update_traces(
    hovertemplate="Product: %{x}<br>Total sales: £%{y:,.0f}<extra></extra>",
    name="Total Sales"
)

fig_top = apply_senior_friendly_layout(
    fig_top,
    f"Top {N_default} Products by Sales – Popular Items for Recommendations"
)

fig_top.update_layout(
    width=1050,
    height=1300,
    margin=dict(l=90, r=140, t=120, b=950),
    title=dict(
        text=f"Top {N_default} Products by Sales – Popular Items for Recommendations",
        font=dict(size=34, family="Trebuchet MS", color=NAVY_DARK),
        x=0.02
    ),
)

fig_top.update_xaxes(
    title_text="",
    tickangle=-45,
    tickfont=dict(size=14, color=NAVY_DARK)
)

fig_top.update_yaxes(
    title_text="Total Sales (£)",
    tickprefix="£",
    separatethousands=True,
    title_font=dict(color=NAVY_DARK),
    tickfont=dict(color=NAVY_DARK)
)

fig_top.add_annotation(
    x=0.5,
    y=-0.9,
    xref="paper",
    yref="paper",
    showarrow=False,
    font=dict(size=20, color=NAVY_DARK),
    text="Product"
)

fig_top.show()

This visualisation identifies the products that generate the highest revenue.
These items act as anchors within the catalogue and are strong candidates for recommendation seeding.
Understanding top-performing products is essential before analysing product relationships and basket behaviour.

In [ ]:
import plotly.express as px

df_tree = (
    df_final
    .groupby("Description", as_index=False)["TotalSales"]
    .sum()
    .sort_values("TotalSales", ascending=False)
    .head(50)
)

fig_tree = px.treemap(
    df_tree,
    path=["Description"],
    values="TotalSales",
    color="TotalSales",
    color_continuous_scale=["#0a1a33", "#12325c", "#4e6a8a"]
)

fig_tree.update_layout(
    title="Product Revenue Treemap – Visualising Sales Concentration",
    title_font=dict(size=28, color="#0a1a33"),
    margin=dict(t=90, l=40, r=40, b=40)
)

fig_tree.show()

The treemap visualises how revenue concentrates across products.
A small number of items account for a significant share of total sales, while many others contribute marginally.
This structure is typical of online retail and supports the use of targeted recommendation strategies rather than uniform promotions.


In [ ]:
import plotly.express as px

min_sup = 0.0005
max_sup = 0.025

rules_viz3d = rules_filtered.copy()
if rules_viz3d.empty:
    rules_viz3d = rules_fp.copy()

# Filter by support range
rules_viz3d = rules_viz3d[
    (rules_viz3d["support"] >= min_sup) &
    (rules_viz3d["support"] <= max_sup)
].copy()

# String versions for hover
rules_viz3d["antecedents_str"] = rules_viz3d["antecedents"].apply(
    lambda x: ", ".join(sorted(list(x)))
)
rules_viz3d["consequents_str"] = rules_viz3d["consequents"].apply(
    lambda x: ", ".join(sorted(list(x)))
)

# Rule length (antecedents + consequents)
rules_viz3d["rule_length"] = (
    rules_viz3d["antecedents"].apply(len) +
    rules_viz3d["consequents"].apply(len)
)

# Bucket for rule length: 2, 3, 4+
def length_class(n):
    if n <= 2:
        return "2 items (1→1)"
    elif n == 3:
        return "3 items"
    else:
        return "4+ items"

rules_viz3d["rule_class"] = rules_viz3d["rule_length"].apply(length_class)

# 3D scatter: color = rule_class, size = lift
fig3d = px.scatter_3d(
    rules_viz3d,
    x="support",
    y="confidence",
    z="lift",
    color="rule_class",          # categorical colors
    size="lift",
    title="3D Rule Map – Support · Confidence · Lift",
    labels={
        "support": "Support",
        "confidence": "Confidence",
        "lift": "Lift",
        "rule_class": "Rule size"
    },
    hover_data={
        "antecedents_str": True,
        "consequents_str": True,
        "support": ":.4f",
        "confidence": ":.3f",
        "lift": ":.2f"
    },
    color_discrete_sequence=["#E63946",  
 "#A8DADC",  
 "#457B9D",  
 "#1D3557"]  

)

# Marker style
fig3d.update_traces(
    marker=dict(
        opacity=0.9,
        line=dict(
            color="rgba(0, 0, 0, 0.25)",
            width=1.0
        )
    )
)

# Background and axes
fig3d.update_layout(
    scene=dict(
        xaxis_title="Support",
        yaxis_title="Confidence",
        zaxis_title="Lift",
        xaxis=dict(backgroundcolor="white", gridcolor="#E5E5E5"),
        yaxis=dict(backgroundcolor="white", gridcolor="#E5E5E5"),
        zaxis=dict(backgroundcolor="white", gridcolor="#E5E5E5")
    ),
    paper_bgcolor="white",
    plot_bgcolor="white"
)

# Short explanation (what low/high means) – in one fluent sentence
fig3d.add_annotation(
    text="Low support = rare rule, high support = common; low confidence = weak, high confidence = reliable; low lift ≈ random, high lift = strong association.",
    xref="paper", yref="paper",
    x=0.50, y=1.10, showarrow=False,
    font=dict(size=11, color="black")
)

# Short button legend under toolbar
fig3d.add_annotation(
    text="Rotate · Pan · Zoom · Reset · Save",
    xref="paper", yref="paper",
    x=0.50, y=-0.17, showarrow=False,
    font=dict(size=10, color="gray")
)

fig3d.show()

It presents a three-dimensional representation of association rules using support, confidence, and lift, allowing a consolidated view of product relationships within customer baskets. The visualisation highlights clusters of rules with high lift and confidence, confirming that several product combinations occur together more frequently than would be expected by chance. These dense areas represent strong behavioural signals, where the purchase of one item significantly increases the likelihood of purchasing another.
As a closing insight, this map validates the relevance of Market Basket Analysis within the overall analytical workflow. The presence of coherent rule clusters confirms that customer purchasing behaviour is structured, repeatable, and suitable for recommendation systems. When combined with temporal patterns and product-level insights shown in previous figures, these associations complete the narrative by linking when customers buy, what they buy, and which products naturally belong together.

### References
https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci